# Chapter 19 - ML Intro Live Coding (From Scratch)

This notebook is a from-scratch companion to an introductory live coding session on machine learning.

It follows the same core structure as the earlier notebook, but does not use any machine learning libraries such as scikit-learn.
All main algorithms and procedures are implemented directly with plain Python plus NumPy and Matplotlib.

## Sections

1. What is Machine Learning
2. Train/Test Split visualization
3. Overfitting demonstration with polynomial regression
4. Decision Trees for classification
5. K-Means clustering
6. Final summary

This notebook is designed for:
- instructor-led live coding
- bachelor-level first exposure to ML
- students running cells alongside in VS Code or Google Colab

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

np.set_printoptions(suppress=True, precision=3)
rng = np.random.default_rng(42)

## 1. What is Machine Learning?

Machine learning means learning patterns from data rather than writing every rule by hand.

### Main subtypes

- Supervised learning
  - regression
  - classification

- Unsupervised learning
  - clustering

### Today

We focus on a few foundational ideas:

- splitting data into training and test parts
- fitting a model
- seeing underfitting and overfitting
- making a simple classifier
- discovering clusters without labels

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Supervised toy example
x_sup = np.linspace(0, 10, 20)
y_sup = 2 * x_sup + 1 + rng.normal(0, 2, size=len(x_sup))
axes[0].scatter(x_sup, y_sup)
axes[0].set_title("Supervised: inputs + answers")
axes[0].set_xlabel("x")
axes[0].set_ylabel("y")

# Unsupervised toy example
cluster1 = rng.normal(loc=[-2, -1], scale=0.6, size=(30, 2))
cluster2 = rng.normal(loc=[2, 2], scale=0.7, size=(30, 2))
cluster3 = rng.normal(loc=[4, -2], scale=0.5, size=(30, 2))
X_unsup = np.vstack([cluster1, cluster2, cluster3])

axes[1].scatter(X_unsup[:, 0], X_unsup[:, 1])
axes[1].set_title("Unsupervised: inputs only")
axes[1].set_xlabel("x1")
axes[1].set_ylabel("x2")

plt.tight_layout()
plt.show()

## 2. Train/Test Split Visualization

A central idea in machine learning is:

- train the model on one part of the data
- test it on different data it has not seen before

Why?

Because a model that only does well on the training data may simply be memorizing.

### From scratch split

We will write our own simple train/test split function:
- shuffle the data
- choose a test fraction
- return training and test subsets

In [ ]:
def train_test_split_from_scratch(X, y, test_size=0.3, seed=42):
    rng_local = np.random.default_rng(seed)
    indices = np.arange(len(X))
    rng_local.shuffle(indices)

    test_count = int(len(X) * test_size)
    test_idx = indices[:test_count]
    train_idx = indices[test_count:]

    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]

X = np.linspace(0, 10, 30).reshape(-1, 1)
y = 3 * X[:, 0] + 5 + rng.normal(0, 3, size=len(X))

X_train, X_test, y_train, y_test = train_test_split_from_scratch(X, y, test_size=0.3, seed=1)

plt.figure(figsize=(8, 4))
plt.scatter(X_train[:, 0], y_train, label="train")
plt.scatter(X_test[:, 0], y_test, label="test", color="red")
plt.title("Train/Test Split From Scratch")
plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.show()

print("Training examples:", len(X_train))
print("Test examples:", len(X_test))

## 3. Overfitting Demonstration with Polynomial Regression

One of the most important ideas in Chapter 19 is the difference between:

- underfitting: model too simple
- good fit: captures the main pattern
- overfitting: follows noise too closely

We will fit polynomial regression from scratch using linear algebra.

### Key idea

A polynomial model like

y = w0 + w1 x + w2 x^2 + ... + wd x^d

is still linear in the parameters w.
That means we can solve it with the normal equation.

### Plan

- create noisy nonlinear data
- build polynomial features ourselves
- solve for weights with NumPy
- compare degree 1, degree 4, and degree 12

In [ ]:
def make_polynomial_features(x, degree):
    x = np.asarray(x).reshape(-1)
    cols = [np.ones_like(x)]
    for d in range(1, degree + 1):
        cols.append(x ** d)
    return np.column_stack(cols)

def fit_polynomial_regression(x, y, degree):
    X_poly = make_polynomial_features(x, degree)
    w = np.linalg.pinv(X_poly) @ y
    return w

def predict_polynomial(x, w):
    degree = len(w) - 1
    X_poly = make_polynomial_features(x, degree)
    return X_poly @ w

x = np.linspace(-3, 3, 30)
y = np.sin(x) + rng.normal(0, 0.25, size=len(x))

x_plot = np.linspace(-3, 3, 400)

degrees = [1, 4, 12]
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)

for ax, degree in zip(axes, degrees):
    w = fit_polynomial_regression(x, y, degree)
    y_pred = predict_polynomial(x_plot, w)
    ax.scatter(x, y, label="data")
    ax.plot(x_plot, y_pred, color="red", label=f"degree {degree}")
    ax.set_title(f"Polynomial degree = {degree}")
    ax.set_xlabel("x")

axes[0].set_ylabel("y")
axes[0].legend()
plt.tight_layout()
plt.show()

### Measuring training and test error

Visual inspection is useful, but we also want numbers.

We will:
- split the nonlinear data into train/test parts
- fit different polynomial degrees
- compare mean squared error

This helps show how a more complex model can improve training error while harming test error.

In [ ]:
def mean_squared_error(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return np.mean((y_true - y_pred) ** 2)

x2 = np.linspace(-3, 3, 60)
y2 = np.sin(x2) + rng.normal(0, 0.25, size=len(x2))

X2 = x2.reshape(-1, 1)
X2_train, X2_test, y2_train, y2_test = train_test_split_from_scratch(X2, y2, test_size=0.35, seed=7)

candidate_degrees = list(range(1, 13))
train_errors = []
test_errors = []

for degree in candidate_degrees:
    w = fit_polynomial_regression(X2_train[:, 0], y2_train, degree)
    train_pred = predict_polynomial(X2_train[:, 0], w)
    test_pred = predict_polynomial(X2_test[:, 0], w)
    train_errors.append(mean_squared_error(y2_train, train_pred))
    test_errors.append(mean_squared_error(y2_test, test_pred))

plt.figure(figsize=(8, 4))
plt.plot(candidate_degrees, train_errors, marker="o", label="train MSE")
plt.plot(candidate_degrees, test_errors, marker="o", label="test MSE")
plt.xlabel("Polynomial degree")
plt.ylabel("Mean squared error")
plt.title("Training vs Test Error")
plt.legend()
plt.show()

## 4. Decision Trees for Classification

A decision tree is a model based on questions.

Example style:
- Is feature 1 less than some threshold?
- If yes, go left
- If no, go right

Eventually we reach a leaf that predicts a class.

### What we will implement

A small binary decision tree classifier from scratch using:

- Gini impurity as the split quality measure
- recursive tree building
- a maximum depth limit for clarity

This is intentionally compact and educational rather than industrial-strength.

In [ ]:
def gini_impurity(y):
    if len(y) == 0:
        return 0.0
    counts = Counter(y)
    probs = [count / len(y) for count in counts.values()]
    return 1.0 - sum(p ** 2 for p in probs)

def best_split(X, y):
    n_samples, n_features = X.shape
    best_feature = None
    best_threshold = None
    best_score = float("inf")

    for feature in range(n_features):
        values = np.unique(X[:, feature])
        if len(values) == 1:
            continue

        thresholds = (values[:-1] + values[1:]) / 2

        for threshold in thresholds:
            left_mask = X[:, feature] <= threshold
            right_mask = ~left_mask

            y_left = y[left_mask]
            y_right = y[right_mask]

            if len(y_left) == 0 or len(y_right) == 0:
                continue

            score = (
                len(y_left) / len(y) * gini_impurity(y_left)
                + len(y_right) / len(y) * gini_impurity(y_right)
            )

            if score < best_score:
                best_score = score
                best_feature = feature
                best_threshold = threshold

    return best_feature, best_threshold, best_score

def majority_class(y):
    return Counter(y).most_common(1)[0][0]

def build_tree(X, y, depth=0, max_depth=3):
    if len(set(y)) == 1:
        return {"type": "leaf", "class": y[0]}

    if depth >= max_depth or len(X) < 2:
        return {"type": "leaf", "class": majority_class(y)}

    feature, threshold, score = best_split(X, y)
    if feature is None:
        return {"type": "leaf", "class": majority_class(y)}

    left_mask = X[:, feature] <= threshold
    right_mask = ~left_mask

    left_tree = build_tree(X[left_mask], y[left_mask], depth + 1, max_depth)
    right_tree = build_tree(X[right_mask], y[right_mask], depth + 1, max_depth)

    return {
        "type": "node",
        "feature": feature,
        "threshold": threshold,
        "left": left_tree,
        "right": right_tree,
    }

def predict_one_tree(tree, x):
    if tree["type"] == "leaf":
        return tree["class"]
    if x[tree["feature"]] <= tree["threshold"]:
        return predict_one_tree(tree["left"], x)
    return predict_one_tree(tree["right"], x)

def predict_tree(tree, X):
    return np.array([predict_one_tree(tree, x) for x in X])

def print_tree(tree, feature_names=None, indent=""):
    if tree["type"] == "leaf":
        print(indent + f"Predict class {tree['class']}")
        return
    fname = f"x[{tree['feature']}]" if feature_names is None else feature_names[tree["feature"]]
    print(indent + f"if {fname} <= {tree['threshold']:.3f}:")
    print_tree(tree["left"], feature_names, indent + "  ")
    print(indent + "else:")
    print_tree(tree["right"], feature_names, indent + "  ")

class0 = rng.normal(loc=[-1.5, -1.0], scale=0.7, size=(25, 2))
class1 = rng.normal(loc=[1.5, 1.2], scale=0.7, size=(25, 2))
X_cls = np.vstack([class0, class1])
y_cls = np.array([0] * len(class0) + [1] * len(class1))

tree = build_tree(X_cls, y_cls, max_depth=3)

print("Learned tree:")
print_tree(tree, feature_names=["x1", "x2"])

preds = predict_tree(tree, X_cls)
accuracy = np.mean(preds == y_cls)
print(f"Training accuracy: {accuracy:.3f}")

### Visualizing the decision tree boundary

Because our data have only two features, we can:
- create a grid of points
- classify every grid point
- color the regions

In [ ]:
def plot_decision_boundary(tree, X, y, title="Decision Tree From Scratch"):
    x1_min, x1_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    x2_min, x2_max = X[:, 1].min() - 1, X[:, 1].max() + 1

    xx1, xx2 = np.meshgrid(
        np.linspace(x1_min, x1_max, 300),
        np.linspace(x2_min, x2_max, 300),
    )

    grid = np.c_[xx1.ravel(), xx2.ravel()]
    zz = predict_tree(tree, grid).reshape(xx1.shape)

    plt.figure(figsize=(6, 5))
    plt.contourf(xx1, xx2, zz, alpha=0.3)
    plt.scatter(X[:, 0], X[:, 1], c=y, edgecolor="black")
    plt.title(title)
    plt.xlabel("x1")
    plt.ylabel("x2")
    plt.show()

plot_decision_boundary(tree, X_cls, y_cls)

## 5. K-Means Clustering

K-means is an unsupervised learning algorithm.

That means:
- we do not give the algorithm labels
- it tries to discover groups in the data

### Core idea

1. Choose k cluster centers
2. Assign each point to its nearest center
3. Recompute each center as the mean of its assigned points
4. Repeat

We will implement this directly from scratch.

In [ ]:
def pairwise_squared_distances(X, centers):
    return np.sum((X[:, None, :] - centers[None, :, :]) ** 2, axis=2)

def kmeans_from_scratch(X, k, max_iters=20, seed=42):
    rng_local = np.random.default_rng(seed)

    init_indices = rng_local.choice(len(X), size=k, replace=False)
    centers = X[init_indices].copy()

    history = [centers.copy()]

    for _ in range(max_iters):
        distances = pairwise_squared_distances(X, centers)
        labels = np.argmin(distances, axis=1)

        new_centers = np.array([
            X[labels == cluster_id].mean(axis=0) if np.any(labels == cluster_id) else centers[cluster_id]
            for cluster_id in range(k)
        ])

        history.append(new_centers.copy())

        if np.allclose(new_centers, centers):
            break

        centers = new_centers

    return centers, labels, history

c1 = rng.normal(loc=[-3, -1], scale=0.6, size=(30, 2))
c2 = rng.normal(loc=[0, 3], scale=0.7, size=(30, 2))
c3 = rng.normal(loc=[3, -2], scale=0.5, size=(30, 2))
X_km = np.vstack([c1, c2, c3])

plt.figure(figsize=(6, 5))
plt.scatter(X_km[:, 0], X_km[:, 1])
plt.title("Unlabeled Data for K-Means")
plt.xlabel("x1")
plt.ylabel("x2")
plt.show()

centers, labels, history = kmeans_from_scratch(X_km, k=3, max_iters=15, seed=3)

plt.figure(figsize=(6, 5))
plt.scatter(X_km[:, 0], X_km[:, 1], c=labels)
plt.scatter(centers[:, 0], centers[:, 1], marker="X", s=250, color="black")
plt.title("K-Means From Scratch")
plt.xlabel("x1")
plt.ylabel("x2")
plt.show()

### Showing the movement of cluster centers

One useful teaching visualization is to show how the centers move during the iterations.

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(X_km[:, 0], X_km[:, 1], alpha=0.35, color="gray")

for cluster_id in range(len(history[0])):
    path = np.array([centers_step[cluster_id] for centers_step in history])
    plt.plot(path[:, 0], path[:, 1], marker="o", label=f"center {cluster_id}")

plt.title("Movement of K-Means Centers")
plt.xlabel("x1")
plt.ylabel("x2")
plt.legend()
plt.show()

print("Number of iterations stored:", len(history) - 1)

## 6. Final Summary

In this notebook, we implemented several foundational machine learning ideas from scratch.

### Main takeaways

- Machine learning learns patterns from data
- Supervised learning includes:
  - regression
  - classification
- Unsupervised learning includes:
  - clustering

### What we built ourselves

- a train/test split
- polynomial regression using linear algebra
- a small decision tree classifier
- k-means clustering

### Important conceptual lessons

- A model can do well on training data but still fail on new data
- More complex models can overfit
- Decision trees make decisions through recursive feature tests
- K-means discovers structure without labels

These examples illustrate the main themes of Chapter 19:
- learning from examples
- generalization
- model complexity
- structure in data